In [1]:
import json
import numpy as np
import torch
import joblib
from tqdm import tqdm
import os
import time
from datetime import datetime
from math import radians, cos, sin, asin, sqrt
from torch.nn.utils.rnn import pad_sequence
from load import haversine
from python_core.general import info, warning, error, debug
import gc

MAX_LEN = 100  # STAN 模型的預設最大軌跡長度

In [2]:
user_lookup = joblib.load("Gowalla_thinned/user_lookup.pkl")
poi_meta = json.load(open("Gowalla_thinned/thinned_poi_metadata.json"))
train_raw = json.load(open("Gowalla_thinned/train_with_timestamps.json"))
test_raw = json.load(open("Gowalla_thinned/test_with_timestamps.json"))

In [3]:
poi_coords = {}
l_max = 0

for k, v in poi_meta.items():
    if "item_id" in v:
        item_id = v["item_id"]
        poi_coords[item_id] = (v["latitude"], v["longitude"])
        l_max = max(l_max, item_id)

info(f"l_max = {l_max}")

INFO   l_max = 29511


In [4]:
train_users = set()

for e in train_raw:
    history = e["history_events"]
    key = (history[0]["raw_location_id"], history[0]["timestamp"])

    if key in user_lookup:
        train_users.add(user_lookup[key])
    else:
        warning(f"Found an unmatched user: {key}")

user_mapping = {uid: idx + 1 for idx, uid in enumerate(train_users)}
u_max = len(user_mapping)

info(f"u_max = {u_max}")

INFO   u_max = 45199


In [ ]:
def time_to_hours(timestamp_str):
    dt = datetime.strptime(timestamp_str, "%Y-%m-%dT%H:%M:%SZ")
    return int(dt.timestamp() / 3600)

def process_and_save_chunks(raw_data, prefix, is_train=True, chunk_size=5000):
    trajs, mat1, mat2t_vec, labels, lens = [], [], [], [], []
    dropped_user = 0
    dropped_poi = 0
    chunk_id = 0
    saved_files = []
    
    # 建立專屬的 chunks 子資料夾
    chunk_dir = 'Gowalla_thinned/chunks'
    os.makedirs(chunk_dir, exist_ok=True)
    
    desc_text = "處理 Train" if is_train else "處理 Test"
    
    for idx, sample in enumerate(tqdm(raw_data, desc=desc_text)):
        history = sample['history_events']
        target = sample['target_event']
        
        # --- 防呆與過濾機制 ---
        key = (history[0]['raw_location_id'], history[0]['timestamp'])
        real_uid = user_lookup.get(key, -1)
        
        if real_uid not in user_mapping:
            if not is_train:
                dropped_user += 1
                mapped_uid = 0
            else:
                continue
        else:
            mapped_uid = user_mapping[real_uid]
        
        all_items = [h['item_id'] for h in history] + [target['item_id']]
        if any(item not in poi_coords for item in all_items):
            if not is_train:
                dropped_poi += 1
            continue

        # --- 長度截斷 ---
        if len(history) > MAX_LEN:
            history = history[-MAX_LEN:]
            
        seq_len = len(history)
        target_time = time_to_hours(target['timestamp'])
        target_item = target['item_id']
        
        # --- 建立軌跡序列 ---
        user_traj = []
        for h in history:
            user_traj.append([mapped_uid, h['item_id'], time_to_hours(h['timestamp'])])
            
        # --- 計算內部時空矩陣 (float32) ---
        init_mat1 = np.zeros((MAX_LEN, MAX_LEN, 2), dtype=np.float32)
        for i in range(seq_len):
            for j in range(seq_len):
                lat1, lon1 = poi_coords[user_traj[i][1]]
                lat2, lon2 = poi_coords[user_traj[j][1]]
                
                init_mat1[i, j, 0] = haversine(lon1, lat1, lon2, lat2)
                init_mat1[i, j, 1] = abs(user_traj[i][2] - user_traj[j][2])
                
        # --- 計算目標時間差向量 (float32) ---
        init_vec = np.zeros((MAX_LEN,), dtype=np.float32)
        for i in range(seq_len):
            init_vec[i] = abs(target_time - user_traj[i][2])
            
        # --- 收集結果 ---
        trajs.append(torch.LongTensor(user_traj))
        mat1.append(init_mat1)
        mat2t_vec.append(init_vec)
        labels.append(target_item)
        lens.append(seq_len)
        
        # ==========================================
        # 核心機制：滿 5000 筆，就存檔並清空記憶體
        # ==========================================
        is_last_item = (idx == len(raw_data) - 1)
        if len(trajs) >= chunk_size or is_last_item:
            trajs_pad = pad_sequence(trajs, batch_first=True, padding_value=0)
            
            chunk_data = [trajs_pad, np.array(mat1, dtype=np.float32), np.array(mat2t_vec, dtype=np.float32), 
                          torch.LongTensor(labels), torch.LongTensor(lens)]
            
            # 寫入 chunks 子資料夾
            chunk_filename = f'{chunk_dir}/{prefix}_chunk_{chunk_id}.pkl'
            joblib.dump(chunk_data, chunk_filename)
            saved_files.append(chunk_filename)
            
            del trajs, mat1, mat2t_vec, labels, lens, chunk_data, trajs_pad
            gc.collect()
            
            trajs, mat1, mat2t_vec, labels, lens = [], [], [], [], []
            chunk_id += 1
            
    if not is_train:
        info(f"Test Set 過濾報告: 丟棄 {dropped_user} 筆(未見過使用者), {dropped_poi} 筆(未見過地點)。")
        
    return saved_files

In [ ]:
start_time = time.time()

candidate_loc = np.linspace(1, l_max, l_max)
mat2s = np.zeros((l_max, l_max))

for i, loc1 in enumerate(tqdm(candidate_loc)):
    for j, loc2 in enumerate(candidate_loc):
        loc1_int, loc2_int = int(loc1), int(loc2)
        if loc1_int in poi_coords and loc2_int in poi_coords:
            lat1, lon1 = poi_coords[loc1_int]
            lat2, lon2 = poi_coords[loc2_int]
            mat2s[i, j] = haversine(lon1, lat1, lon2, lat2)

joblib.dump(mat2s, "mat2s.pkl")

print(f"計算完成！耗時: {time.time() - start_time:.2f} 秒")
print(f"全局矩陣大小: {mat2s.shape}")

100%|██████████| 29511/29511 [26:10<00:00, 18.79it/s]


計算完成！耗時: 1583.11 秒
全局矩陣大小: (29511, 29511)


In [6]:
mat2s = joblib.load('mat2s.pkl')

In [7]:
start_time = time.time()

train_chunk_files = process_and_save_chunks(
    train_raw, 
    prefix='train_stan', 
    is_train=True,
    chunk_size=5000
)
print(f"Train 分塊計算完成！共安全產出 {len(train_chunk_files)} 個暫存檔。")

print("\n打包全域資訊 (Global Info)...")
global_info = {
    'mat2s': np.array(mat2s, dtype=np.float32),
    'u_max': u_max,
    'l_max': l_max
}
global_path = 'Gowalla_thinned/global_info.pkl'
joblib.dump(global_info, global_path)

print(f"全域資訊已獨立儲存至: {global_path}")
print("Train 處理完美收工！(請保留 chunks 資料夾內的所有檔案)")
print(f"總耗時: {time.time() - start_time:.2f} 秒")

處理 Train: 100%|██████████| 675267/675267 [10:15<00:00, 1096.91it/s] 


Train 分塊計算完成！共安全產出 136 個暫存檔。

打包全域資訊 (Global Info)...
全域資訊已獨立儲存至: Gowalla_thinned/global_info.pkl
Train 處理完美收工！(請保留 chunks 資料夾內的所有檔案)
總耗時: 662.63 秒


In [10]:
start_time = time.time()

test_chunk_files = process_and_save_chunks(
    test_raw, 
    prefix='test_stan', 
    is_train=False, 
    chunk_size=5000
)

print(f"Test 分塊計算完成！共安全產出 {len(test_chunk_files)} 個暫存檔。")
print("Test 處理完美收工！(請保留 chunks 資料夾內的所有檔案)")
print(f"總耗時: {time.time() - start_time:.2f} 秒")

處理 Test: 100%|██████████| 155132/155132 [01:58<00:00, 1311.54it/s]

INFO   Test Set 過濾報告: 丟棄 44883 筆(未見過使用者), 0 筆(未見過地點)。
Test 分塊計算完成！共安全產出 22 個暫存檔。
Test 處理完美收工！(請保留 chunks 資料夾內的所有檔案)
總耗時: 118.30 秒


In [9]:
dropped_because_lookup_failed = 0
dropped_because_user_unseen = 0
total_test_sessions = len(test_raw)

print("=== Test Set 丟棄原因診斷報告 ===")

for sample in test_raw:
    history = sample['history_events']
    # 取出查詢用的 Key
    key = (history[0]['raw_location_id'], history[0]['timestamp'])
    
    # 嘗試反查真實 UID
    real_uid = user_lookup.get(key, -1)
    
    if real_uid == -1:
        # 原因 A：根本查不到這個人是誰
        dropped_because_lookup_failed += 1
    elif real_uid not in user_mapping:
        # 原因 B：查得到是誰，但他確實沒出現在 Train 裡面
        dropped_because_user_unseen += 1

print(f"Test Set 總 Session 數: {total_test_sessions}")
print(f"[原因 A] 字典反查失敗 (找不到紀錄): {dropped_because_lookup_failed} 筆")
print(f"[原因 B] 確實是未見過的使用者 (Train沒看過): {dropped_because_user_unseen} 筆")
print(f"成功存活的有效 Test Session: {total_test_sessions - dropped_because_lookup_failed - dropped_because_user_unseen} 筆")

# 如果有反查失敗的，印出前 3 筆來看看長怎樣
if dropped_because_lookup_failed > 0:
    print("\n[抽樣檢查] 查不到的 Key 範例 (raw_location_id, timestamp):")
    fail_count = 0
    for sample in test_raw:
        history = sample['history_events']
        key = (history[0]['raw_location_id'], history[0]['timestamp'])
        if user_lookup.get(key, -1) == -1:
            print(f"  -> {key}")
            fail_count += 1
            if fail_count >= 3:
                break

=== Test Set 丟棄原因診斷報告 ===
Test Set 總 Session 數: 155132
[原因 A] 字典反查失敗 (找不到紀錄): 0 筆
[原因 B] 確實是未見過的使用者 (Train沒看過): 44883 筆
成功存活的有效 Test Session: 110249 筆
